# Specification Sensitivity: Comparing Four England NMF Models

Compares nmf_eng_5, nmf_eng_15, nmf_eng_30, and nmf_eng_30_nm to demonstrate how specification choices (topic count, source composition) shape AI outputs.

| Model | k | Corpus | Purpose |
|-------|---|--------|--------|
| nmf_eng_5 | 5 | Full | Source-proxy collapse demonstration |
| nmf_eng_15 | 15 | Full | Midpoint — where minority sources become audible |
| nmf_eng_30 | 30 | Full | Production model — full thematic resolution |
| nmf_eng_30_nm | 30 | No SchoolsWeek | Source composition sensitivity |

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

models = {}
for model_id in ["nmf_eng_5", "nmf_eng_15", "nmf_eng_30", "nmf_eng_30_nm"]:
    path = f"/workspaces/AM1_topic_modelling/data/evaluation_outputs/{model_id}_summary.json"
    try:
        with open(path) as f:
            models[model_id] = json.load(f)
        print(f"Loaded {model_id}: {models[model_id]['n_topics']} topics, {models[model_id]['n_articles']} articles")
    except FileNotFoundError:
        print(f"NOT FOUND: {path} — run the notebook first")

print(f"\nModels loaded: {len(models)}/4")

## 1. Cross-k metrics comparison

In [ ]:
rows = []
for model_id, data in models.items():
    m = data["metrics"]
    topics = data["topics"]
    single_source = sum(1 for t in topics if t.get("single_source", False))
    largest_pct = max(t["pct"] for t in topics)
    smallest_pct = min(t["pct"] for t in topics)
    
    rows.append({
        "model": model_id,
        "k": data["n_topics"],
        "articles": data["n_articles"],
        "recon_error": m["reconstruction_error"],
        "stability": m["stability"],
        "mean_weight": m["mean_dominant_weight"],
        "max_weight": m["max_dominant_weight"],
        "largest_topic_pct": largest_pct,
        "smallest_topic_pct": smallest_pct,
        "single_source_topics": f"{single_source}/{data['n_topics']}",
    })

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))

## 2. Topic distribution comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, (model_id, data) in enumerate(models.items()):
    ax = axes[idx // 2][idx % 2]
    topics = data["topics"]
    names = [t["name"] for t in topics]
    pcts = [t["pct"] for t in topics]
    
    bars = ax.barh(range(len(names)), pcts)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=7)
    ax.set_xlabel("% of corpus")
    ax.set_title(f"{model_id} (k={data['n_topics']}, n={data['n_articles']})")
    ax.invert_yaxis()

plt.suptitle("Topic Distribution Across Four Models", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("/workspaces/AM1_topic_modelling/data/evaluation_outputs/4model_topic_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Source concentration comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

model_names = []
single_counts = []
multi_counts = []
total_topics = []

for model_id, data in models.items():
    topics = data["topics"]
    single = sum(1 for t in topics if t.get("single_source", False))
    multi = len(topics) - single
    model_names.append(f"{model_id}\n(k={data['n_topics']})")
    single_counts.append(single)
    multi_counts.append(multi)
    total_topics.append(len(topics))

x = range(len(model_names))
ax.bar(x, single_counts, label="Single-source (>90%)", color="coral")
ax.bar(x, multi_counts, bottom=single_counts, label="Multi-source", color="steelblue")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel("Number of topics")
ax.set_title("Single-source vs multi-source topics across models")
ax.legend()

for i, (s, t) in enumerate(zip(single_counts, total_topics)):
    ax.text(i, t + 0.3, f"{s}/{t} ({s/t*100:.0f}%)", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("/workspaces/AM1_topic_modelling/data/evaluation_outputs/4model_source_concentration.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for model_id, data in models.items():
    print(f"\n{'='*80}")
    print(f"  {model_id} (k={data['n_topics']})")
    print(f"{'='*80}")
    for t in data["topics"]:
        marker = " <<<" if t.get("single_source", False) else ""
        print(f"  {t['name']:<45} {t['top_source']:<15} {t['top_source_pct']:.0%}{marker}")

## 4. Dominant weight comparison — model confidence across k

In [ ]:
weight_data = []
for model_id, data in models.items():
    weight_data.append({
        "model": model_id,
        "k": data["n_topics"],
        "mean_weight": data["metrics"]["mean_dominant_weight"],
        "max_weight": data["metrics"]["max_dominant_weight"],
    })

weight_df = pd.DataFrame(weight_data)

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(weight_df))
ax.bar([i - 0.15 for i in x], weight_df["mean_weight"], width=0.3, label="Mean", color="steelblue")
ax.bar([i + 0.15 for i in x], weight_df["max_weight"], width=0.3, label="Max", color="coral")
ax.set_xticks(x)
ax.set_xticklabels([f"{r['model']}\n(k={r['k']})" for _, r in weight_df.iterrows()])
ax.set_ylabel("Dominant topic weight")
ax.set_title("Model confidence: mean and max dominant weight across k")
ax.legend()
plt.tight_layout()
plt.savefig("/workspaces/AM1_topic_modelling/data/evaluation_outputs/4model_weights.png", dpi=150, bbox_inches="tight")
plt.show()

print(weight_df.to_string(index=False))

## 5. What changes between k=5 and k=30? — Topic splitting

In [ ]:
k5_topics = {t["name"]: t for t in models.get("nmf_eng_5", {}).get("topics", [])}
k15_topics = {t["name"]: t for t in models.get("nmf_eng_15", {}).get("topics", [])}
k30_topics = {t["name"]: t for t in models.get("nmf_eng_30", {}).get("topics", [])}

if k5_topics:
    print("How k=5 topics split at k=15 and k=30:")
    print("=" * 80)
    
    splits = {
        "pupil_welfare_and_inclusion (k=5, 49.3%)": {
            "k=15": ["child_welfare_and_early_years", "pupil_attainment_and_disadvantage", 
                     "send_and_council_deficits", "school_absence_and_attendance"],
            "k=30": ["child_and_family_welfare", "pupil_disadvantage_and_attainment",
                     "send_and_council_deficits", "school_absence_and_attendance",
                     "free_school_meals_and_poverty", "mental_health_and_wellbeing",
                     "exclusions_and_suspensions", "primary_assessment_and_sats"],
        },
        "teacher_pay_and_workforce (k=5)": {
            "k=15": ["teacher_recruitment_and_retention", "teacher_pay_and_strikes"],
            "k=30": ["teacher_pay", "teacher_strikes_and_unions", "teacher_recruitment_and_retention"],
        },
        "academy_financial_oversight (k=5)": {
            "k=15": ["academy_financial_oversight"],
            "k=30": ["academy_finance_and_oversight"],
        },
        "academy_trust_governance (k=5)": {
            "k=15": ["academy_trust_governance", "dfe_intervention_notices"],
            "k=30": ["academy_trust_governance", "dfe_warning_notices"],
        },
        "ofsted_inspection_reform (k=5)": {
            "k=15": ["ofsted_inspection_reform"],
            "k=30": ["ofsted_inspection_reform", "ofsted_report_cards"],
        },
    }
    
    for parent, children in splits.items():
        print(f"\n{parent}")
        for k_level, topics in children.items():
            print(f"  {k_level}: {', '.join(topics)}")

## 6. k=30 vs k=30 no media — what changes when SchoolsWeek is removed?

In [ ]:
if "nmf_eng_30" in models and "nmf_eng_30_nm" in models:
    full = {t["name"]: t for t in models["nmf_eng_30"]["topics"]}
    nm = {t["name"]: t for t in models["nmf_eng_30_nm"]["topics"]}
    
    print(f"Full corpus: {models['nmf_eng_30']['n_articles']} articles, 30 topics")
    print(f"No media:    {models['nmf_eng_30_nm']['n_articles']} articles, 30 topics")
    print(f"\nFull corpus topics: {len(full)}")
    print(f"No media topics:    {len(nm)}")
    
    # Topics in full but not in nm (by name)
    full_names = set(full.keys())
    nm_names = set(nm.keys())
    
    print(f"\nTopics only in full corpus: {full_names - nm_names}")
    print(f"Topics only in no-media:    {nm_names - full_names}")
    print(f"Shared topic names:         {len(full_names & nm_names)}")
    
    print(f"\nNo-media topics:")
    for t in sorted(models["nmf_eng_30_nm"]["topics"], key=lambda x: x["pct"], reverse=True):
        print(f"  {t['name']:<45} {t['count']:>4} ({t['pct']}%)  top: {t['top_source']} {t['top_source_pct']:.0%}")
else:
    print("nmf_eng_30_nm not loaded — run the no-media notebook first")

## 7. Summary: specification sensitivity across all four models

In [ ]:
print("=" * 80)
print("SPECIFICATION SENSITIVITY SUMMARY — FOUR ENGLAND MODELS")
print("=" * 80)

for model_id, data in models.items():
    m = data["metrics"]
    topics = data["topics"]
    single = sum(1 for t in topics if t.get("single_source", False))
    largest = max(topics, key=lambda t: t["pct"])
    
    print(f"\n{model_id} (k={data['n_topics']}, n={data['n_articles']})")
    print(f"  Stability:        {m['stability']:.4f}")
    print(f"  Mean weight:      {m['mean_dominant_weight']:.4f}")
    print(f"  Largest topic:    {largest['name']} ({largest['pct']}%)")
    print(f"  Single-source:    {single}/{data['n_topics']}")

print(f"\n{'='*80}")
print("Key finding: same data, same pipeline, different k → different findings.")
print("Same k, different data (media removed) → different findings.")
print("Standard quality metrics do not flag these differences.")
print(f"{'='*80}")